In [ ]:
import pydataset
mpg = pydataset.data('mpg')

In [ ]:
mpg['fl'].value_counts()

In [ ]:
mpg.drop(index = mpg[(mpg['fl'] == 'e')].index, inplace=True)
mpg.drop(index = mpg[mpg['fl'] == 'd'].index, inplace=True)
mpg.drop(index = mpg[mpg['fl'] == 'c'].index, inplace=True)
mpg['fl'].value_counts()

In [ ]:
X_features = mpg.drop(['fl'], axis = 1)
y_labels = mpg['fl'] 

In [ ]:
# ?? ???? ???? ???? ?? ??? ??? ???
X_features.info()

# <class 'pandas.core.frame.DataFrame'>
# Int64Index: 220 entries, 1 to 234
# Data columns (total 10 columns):
#  #   Column        Non-Null Count  Dtype  
# ---  ------        --------------  -----  
#  0   manufacturer  220 non-null    object 
#  1   model         220 non-null    object 
#  2   displ         220 non-null    float64
#  3   year          220 non-null    int64  
#  4   cyl           220 non-null    int64  
#  5   trans         220 non-null    object 
#  6   drv           220 non-null    object 
#  7   cty           220 non-null    int64  
#  8   hwy           220 non-null    int64  
#  9   class         220 non-null    object 
# dtypes: float64(1), int64(4), object(5)
# memory usage: 18.9+ KB

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y_labels = encoder.fit_transform(y_labels)
y_labels

In [ ]:
columns = X_features.columns

for column in ['manufacturer', 'model', 'trans', 'drv', 'class']:
    X_features[column] = encoder.fit_transform(X_features[column])

X_features

In [ ]:
# ????
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
import numpy as np

dt_clf = DecisionTreeClassifier()

scores = cross_val_score(dt_clf, X_features, y_labels, scoring = 'accuracy', cv = 5)
print(f'??? : {np.mean(scores)}')

In [ ]:
import pandas as pd
import numpy as np

# ????
card_df = pd.read_csv("./creditcard.csv/creditcard.csv")
card_df[:3]

In [ ]:
card_df.shape

## ?? ???? ?? ???? ??
???? ???? ??? ?? Data set? ?? ?? ? ????? ??? ??? ? ??. ?? ???? ??? ??? ??? ?? ?? ??? ??? ??? ??? ???? ??? ?? ?? ???? ??? ??? ??? ?? ?? ??? ??? ????. ??? ??? ??? ?? ???? ???? ??? ????, ????? ?? ???? ?? ???? ??.

### ?? ???
- ?? ??? ?? ?? ??? ? ???? ????? ??
- ?? ???? 10,000?, ?????? 100 ???? ?? ??? ???? 100??? ?????
- ???? ?? ???? ??/???? ???? ?????? ??? ??? ??? ??? ?? ???? ???? ?? ??? ??? ? ??.

### ?? ???
- ?? ??? ? ??? ??? ?? ?? ????
- ?? ??? ?? ?? ??? ? ???? ???? ??? ??
- ??? ?? ???? ????? ??? ? ? ?? ???, ?? ??? ???? ?? ???? ????
- ?????? SMOTE(Synthetic Minority Over-sampling Technique) ??? ??
- SMOTE ? ?? ??? ?? ?? ?? ????? K??? ??(KNN)? ?? ???? K?? ??? ??? ?? ??? ??? ?? ???? ??? ??? ?? ??? ????? ???? ????.

In [ ]:
# Time ??? ?? ??? ?? ????.
card_df.drop(['Time'], axis=1, inplace=True)
card_df

In [ ]:
# ??? ? ??
from sklearn.model_selection import train_test_split

X_features = card_df.iloc[:,:-1]
y_lables = card_df.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X_features, y_lables, test_size=0.3, random_state=0, stratify=y_lables)

stratify?

In [ ]:
# ??? ??? ???? ??? ?? ???? ?? ?? ??
print('????? ??? ? ??')
print(y_train.value_counts()/len(y_train))
print()
print('??? ??? ??? ? ??')
print(y_test.value_counts()/len(y_test))

?? ???? ??? ???? 1?? ? 0.0017? ????? ????? ??? ? ? ??.

?? ???? ??? ???? ?? ??? ??????. ?? ??? ?? ???? ??? ?? `get_clf_eval()` ??? ?? ??

In [ ]:
# ?? ??
from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression(max_iter = 1000)

# ??/??
lr_clf.fit(X_train, y_train)
preds = lr_clf.predict(X_test)

# output? ??? ???? ?( =1? =positive?)? ????.
lr_pred_proba = lr_clf.predict_proba(X_test)[:,1]

In [ ]:
# ??? ????
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC ?? 
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('?? ??')
    print(confusion, '\n')
    # ROC-AUC print ??
    print('???: {0:.4f}, ???: {1:.4f}, ???: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc)) 

In [ ]:
# ??
get_clf_eval(y_test, preds, lr_pred_proba)

???? LGBMClassifier? ???? ??? ??? ? ??? ??? ?????? ?? ??? ?????

In [ ]:
# ?? ??, fit, predict, evaluate
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
lgbm_clf.fit(X_train, y_train)
preds = lgbm_clf.predict(X_test)
lgbm_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]

# ????, ???, ???, ???, f1, auc
get_clf_eval(y_test, preds, lgbm_pred_proba)

## ????? ??? ?? ? ??
### ?? ???? ??? ??

In [ ]:
card_df['Amount'].describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.xticks(range(0, 30000, 1000), rotation=60)
sns.histplot(card_df['Amount'], bins=100, kde=True)
plt.show()

In [ ]:
# ??? ??
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
card_df['Amount'] = scaler.fit_transform(card_df['Amount'].values.reshape(-1, 1))

In [ ]:
card_df['Amount'].max()

In [ ]:
card_df['Amount'].min()

In [ ]:
# ??? ?? ??? 0-??? 1? ??? ?? ????? ???? ??
print(f"??\n{card_df['Amount'].mean()}")
print(f"??\n{card_df['Amount'].var()}")

In [ ]:
# ??? ? ??
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_features, y_lables, test_size=0.3, random_state=0, stratify=y_lables)

# ???? ?? ??
lr_clf = LogisticRegression(max_iter = 1000)
lr_clf.fit(X_train, y_train)
preds = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:,1]
# ??
print('???? ?? ??')
get_clf_eval(y_test, preds, lr_pred_proba)


# LightGBM ?? ??
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
lgbm_clf.fit(X_train, y_train)
preds = lgbm_clf.predict(X_test)
lgbm_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]
# ??
print('\nLightGBM ?? ??')
get_clf_eval(y_test, preds, lgbm_pred_proba)

?? ? ??
```
???? ?? ??
[???: 0.9992, ???: 0.8667, ???: 0.6149,    F1: 0.7194, AUC:0.9702]
LightGBM ?? ??
[???: 0.9995, ???: 0.9573, ???: 0.7568,    F1: 0.8453, AUC:0.9790]
```
?? ????? ?? ???? ???? ?? ??? ? ??. ~~???? ??? ??? ?? ??~~

## ?? ?? ? ??

?? ??? ??? ???? ??? ?? ???? ?? ???? ?? ?? ? ??. ?? ?? log??? ??? ?? ??? ???? ??? ??? ???? ??? ?? ?? ??? ??.

In [ ]:
card_df_log = pd.read_csv("./creditcard.csv/creditcard.csv")
card_df_log.drop(['Time'], axis=1, inplace=True)
card_df_log[:3]

In [ ]:
# log ????
# numpy ? log1p()??? ??? ?????
card_df_log['Amount'] = np.log1p(card_df_log['Amount'])

In [ ]:
# ??? ? ??
from sklearn.model_selection import train_test_split
X_features = card_df_log.iloc[:,:-1]
y_lables = card_df_log.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X_features, y_lables, test_size=0.3, random_state=0, stratify=y_lables)

# ???? ?? ??
lr_clf = LogisticRegression(max_iter = 1000)
lr_clf.fit(X_train, y_train)
preds = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:,1]
# ??
print('???? ?? ??')
get_clf_eval(y_test, preds, lr_pred_proba)


# LightGBM ?? ??
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
lgbm_clf.fit(X_train, y_train)
preds = lgbm_clf.predict(X_test)
lgbm_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]
# ??
print('\nLightGBM ?? ??')
get_clf_eval(y_test, preds, lgbm_pred_proba)

?? ? ??
```
???? ?? ??
[???: 0.9992, ???: 0.8667, ???: 0.6149,    F1: 0.7194, AUC:0.9702]
LightGBM ?? ??
[???: 0.9995, ???: 0.9573, ???: 0.7568,    F1: 0.8453, AUC:0.9790]
```
????? ?? ???? AUC? ?? ?????? ???? ??? ??.
LightGBM? ??? ???? ?? ?? ??? ??? ??.

## ??? ?? ? ??
???? ?? ??? ????? ??? ???? IQR(Inter Quantile Range) ??? ????. ?? ???? ???? ???? ?? ??? ?? ????, ???? ???? ?? ?? ???? ??? ???? ?????? ?? ?? ??? ???? ?? ??? ???(? ???)? ?? ???? ?? ???? ??? ???? ???? ?? ??.

In [ ]:
card_df_iqr = pd.read_csv("./creditcard.csv/creditcard.csv")
card_df_iqr.drop(['Time'], axis=1, inplace=True)
card_df_iqr[:3]

In [ ]:
# DataFrame? corr()? ??? ? ???? ???? ?? ? heatmap ???
# ?? ???? ???? ???? ????, ?? ????? ?? ?? ?? ?? ???.
corr_M = card_df_iqr.corr()
sns.heatmap(corr_M, cmap="RdBu")
plt.show()

???? ?? ?? ????? ?? ?? ?? v14? v17? ??? ?? v14? ???? ???? ?? ?????? ??.

In [ ]:
# class ? 1? ???? ??? ??? ??? ????.
fraud = card_df_iqr[card_df_iqr['Class'] == 1]
fraud

In [ ]:
# ??? ??? ??? 1/4??? 3/4??? ???, ?? ???? IQR? ????.
Q_25 = np.percentile(fraud.V14, 25)
Q_75 = np.percentile(fraud.V14, 75)
IQR = Q_75 -Q_25
IQR

In [ ]:
# ? ??? IQR ????
Q_25 = fraud['V14'].quantile(.25)
Q_75 = fraud['V14'].quantile(.75)
IQR = Q_75 - Q_25
IQR

In [ ]:
# ??? IQR? 1.5? ?? ???? ??? ?? ???
upper_out = Q_75 + 1.5 * IQR
under_out = Q_25 - 1.5 * IQR

# ??? ??? ?? 
cond1 = fraud['V14'] >= upper_out
cond2 = fraud['V14'] <= under_out
fraud.V14[cond1 | cond2]

In [ ]:
# ???? ??? ??? ????
temp = fraud.V14[cond1 | cond2].index
temp

In [ ]:
card_df_iqr.shape

In [ ]:
# card_df?? ???? ??????
card_df_iqr.drop(labels = temp, axis = 0, inplace=True)

In [ ]:
card_df_iqr.shape

?? ? 'shape (284807, 30)'?? ??? 4? ???? 'shape (284803, 30)' ? ? ??? ?? ??? ? ??.

In [ ]:
# ??? ? ??
from sklearn.model_selection import train_test_split
X_features = card_df_iqr.iloc[:,:-1]
y_lables = card_df_iqr.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X_features, y_lables, test_size=0.3, random_state=0, stratify=y_lables)

# ???? ?? ??
lr_clf = LogisticRegression(max_iter = 1000)
lr_clf.fit(X_train, y_train)
preds = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:,1]
# ??
print('???? ?? ??')
get_clf_eval(y_test, preds, lr_pred_proba)


# LightGBM ?? ??
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
lgbm_clf.fit(X_train, y_train)
preds = lgbm_clf.predict(X_test)
lgbm_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]
# ??
print('\nLightGBM ?? ??')
get_clf_eval(y_test, preds, lgbm_pred_proba)

?? ? ??
```
???? ?? ??
[???: 0.9992, ???: 0.8667, ???: 0.6149,    F1: 0.7194, AUC:0.9702]
LightGBM ?? ??
[???: 0.9995, ???: 0.9573, ???: 0.7568,    F1: 0.8453, AUC:0.9790]
```
????? ?? ???? ???, f1, auc ? ?? ???? ???? ?? ??? ??
LightGBM? ?? 5?? ?? ????? ??? ??? ?? ??

## log? ??? ??? ?? ????

In [ ]:
# card_df_iqr ???? log ??
card_df_iqr['Amount'] = np.log1p(card_df_iqr['Amount'])

In [ ]:
# ??? ? ??
from sklearn.model_selection import train_test_split
X_features = card_df_iqr.iloc[:,:-1]
y_lables = card_df_iqr.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X_features, y_lables, test_size=0.3, random_state=0, stratify=y_lables)

# ???? ?? ??
lr_clf = LogisticRegression(max_iter = 1000)
lr_clf.fit(X_train, y_train)
preds = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:,1]
# ??
print('???? ?? ??')
get_clf_eval(y_test, preds, lr_pred_proba)


# LightGBM ?? ??
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
lgbm_clf.fit(X_train, y_train)
preds = lgbm_clf.predict(X_test)
lgbm_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]
# ??
print('\nLightGBM ?? ??')
get_clf_eval(y_test, preds, lgbm_pred_proba)

??? ??? ?? ?? ??? ???.
```
???? ?? ??
[???: 0.9993, ???: 0.8596, ???: 0.6712,    F1: 0.7538, AUC:0.9739]

LightGBM ?? ??
[???: 0.9997, ???: 0.9680, ???: 0.8288,    F1: 0.8930, AUC:0.9791]
```
?? ?? ?.. ????? ?? ???? f1, auc? ?????.
LightGBM? ????? ?? ??.
??? ????? ???? ?? ?? ?? ???? ?? ???? ??? ??? ???.

## SMOTE ?? ??? ??
SMOTE? ??? ??? ??? ?? ??? ?? ?? ???? ????. ?? ??? ??? ??? ??? ?? ?? ???? ?? ??? ?? ??? ??? ?? ??? ???? ?? ?? ???? ???? ??? ??? ??/???? ?? ?? ????.

In [ ]:
# SMOTE? ??? ???? ??? ??? ??
!pip install imbalanced-learn

In [ ]:
# ??? ??
from imblearn.over_sampling import SMOTE

# test? ???? ???. ??? train? ????
smote = SMOTE(random_state=0)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f'SMOTE ??? ??? ??/??? ??? ? : {X_train.shape}, {y_train.shape}')
print(f'SMOTE ??? ??? ??/??? ??? ? : {X_train_smote.shape}, {y_train_smote.shape}')
print(f'SMOTE ??? ??? ? ?? : \n{pd.Series(y_train_smote).value_counts()}')

In [ ]:
# ???? ?? ??
lr_clf = LogisticRegression(max_iter = 1000)
lr_clf.fit(X_train, y_train)
preds = lr_clf.predict(X_test)
lr_pred_proba = lr_clf.predict_proba(X_test)[:,1]
# ??
print('???? ?? ??')
get_clf_eval(y_test, preds, lr_pred_proba)


# LightGBM ?? ??
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
lgbm_clf.fit(X_train, y_train)
preds = lgbm_clf.predict(X_test)
lgbm_pred_proba = lgbm_clf.predict_proba(X_test)[:,1]
# ??
print('\nLightGBM ?? ??')
get_clf_eval(y_test, preds, lgbm_pred_proba)

SMOTE ? ? ??
```
???? ?? ??
[???: 0.9992, ???: 0.8667, ???: 0.6149,    F1: 0.7194, AUC:0.9702]
LightGBM ?? ??
[???: 0.9995, ???: 0.9573, ???: 0.7568,    F1: 0.8453, AUC:0.9790]
```
????? ?? ????? ??? ??? ??
LightGBM? ???? ????? ??? ??? ??? ???.

In [ ]:
def get_model_train_eval(model, train_x=None, test_x=None, train_y=None, test_y=None):
    model.fit(train_x, train_y)
    pred = model.predict(test_x)
    pred_proba = model.predict_proba(test_x)[:,1]
    get_clf_eval(test_y, pred, pred_proba)

In [ ]:
print('LightGBM : boost_from_average=True ?? ??')
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=True)
get_model_train_eval(lgbm_clf, train_x=X_train, test_x=X_test, train_y=y_train, test_y=y_test)



print('\nLightGBM ?? ??')
lgbm_clf = LGBMClassifier(n_estimators=1000, num_leaves=64, n_jobs=-1, boost_from_average=False)
get_model_train_eval(lgbm_clf, train_x=X_train, test_x=X_test, train_y=y_train, test_y=y_test)


# ?? ???? True / False? ????? ??? ?? ??? ?? ??? ?? ??? ? ???. ??? ???? ? ????? ???? ?? ?????? ???? ????? ??? ???? ???? ???(? ?????...

LightGBM 2.1.0 ?? ???? `boost_from_average` ???? ??? ?? True?? ??? ??? ?? ????? False ? ??? ?? ?? ??.

In [ ]:
LGBMClassifier?